In [1]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
import json
from typing import List
wor_dir = "/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis"
os.chdir(wor_dir)
sys.path.append("../../benchmark")
import test_base
from sentence_splitter import split_text_into_sentences
from BERT_classifier.Classify_report_with_BERT import classification_report_BERT
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import preprocessing

In [2]:
sentence_length = 6

## Test using a BERT-Classification Model trained on paragraphs to classify an entire report.

**Function:** pdf -> NACE Class

In [3]:
dataset_path = "data/datasets/german_annual_reports"
dataset_path = "data/datasets/stoxx_600_extended"
dataset_path = "data/datasets/reports_subset_from_full_data_1"
dataset_path = "data/datasets/reports_subset_from_full_data_3"
dataset_path = "data/datasets/stoxx_600"

In [4]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [5]:
nace_classes = pd.read_csv(over_view_df_path, index_col=0, sep=";")
nace_classes.head()

,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report,description_page
483,3i Group plc,III-GB,III-GB,701.210162,1276.962137,NaN,66.30,K,NaN,NaN
354,A.P. Moller - Maersk A/S Class B,MAERSK.B-DK,MAERSK.B-DK,77568.022858,47226.219381,51288.2262335699,50.20,H,NaN,NaN
275,A2A S.p.A.,A2A-IT,A2A-IT,22938.000000,14492.000000,NaN,35.11,D,NaN,NaN
107,AAK AB,AAK-SE,AAK-SE,4741.086317,4010.433824,3939.34280627966,10.89,C,AAK AB1.pdf,3
122,Aalberts N.V.,AALB-NL,AALB-NL,3230.000000,3324.000000,3148.6,25.93,C,Aalberts N.V.1.pdf,NaN


In [6]:

nace_classes

,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report,description_page
483,3i Group plc,III-GB,III-GB,701.210162,1276.962137,NaN,66.30,K,NaN,NaN
354,A.P. Moller - Maersk A/S Class B,MAERSK.B-DK,MAERSK.B-DK,77568.022858,47226.219381,51288.2262335699,50.20,H,NaN,NaN
275,A2A S.p.A.,A2A-IT,A2A-IT,22938.000000,14492.000000,NaN,35.11,D,NaN,NaN
107,AAK AB,AAK-SE,AAK-SE,4741.086317,4010.433824,3939.34280627966,10.89,C,AAK AB1.pdf,3
122,Aalberts N.V.,AALB-NL,AALB-NL,3230.000000,3324.000000,3148.6,25.93,C,Aalberts N.V.1.pdf,NaN
...,...,...,...,...,...,...,...,...,...,...
577,WPP Plc,WPP-GB,WPP-GB,16910.961740,17068.346685,17413.6933169365,73.11,M,NaN,NaN
114,Yara International ASA,YAR-NO,YAR-NO,22742.705923,14273.219511,12820.6705029309,20.15,C,Yara International ASA2.pdf,NaN
344,Zalando SE,ZAL-DE,ZAL-DE,10344.800000,10143.100000,10572.5,47.91,G,NaN,NaN
246,Zealand Pharma A/S,ZEAL-DK,ZEAL-DK,13.977698,46.005625,8.40500161716342,21.20,C,NaN,NaN


In [7]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
len(report_to_nace_class)

291

In [8]:
reports_path = glob.glob(os.path.join(dataset_path, "PDFs/*.pdf"))
len(reports_path)

261

In [9]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
len(reports_path)

263

In [10]:
## ! Only filter the description pages !

nace_classes_description_pages = nace_classes.dropna(subset="description_page")
nace_classes_description_pages
description_page_path = "data/datasets/stoxx_600/company_descriptions_txt/"
reports_path = [description_page_path + name.replace("pdf", "txt") for name in nace_classes_description_pages["Report"].to_list()]

In [11]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")
    lines = [line for line in lines if line != ""]

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        #lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        #lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks
    
    if len(chunks) == 0: 
        return []
    elif len(chunks) == 1: 
        return chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [12]:
model_version = "1_0"
model_version = "2_1"
model_version = "1_1"
model_version = "2_2"

if model_version == "1_0":
    ckpt = wor_dir + "/results/BERT_models/results_null_classifiers__cos_thres_0.5__bert-base-uncased__train_full_model__some_labels/checkpoint-2331"
    num_layers = 1
elif model_version == "1_1":
    ckpt = wor_dir + "/results/BERT_models/results__new_approach_data__num_layers_2__cos_thres_0.5bert-base-uncased__train_full_model__some_labels__only_labels/checkpoint-6240"
    num_layers = 2
elif model_version == "2_1":
    ckpt = wor_dir + "/results/BERT_models/_best_2nd_results__new_approach_data__num_layers_2__cos_thres_0.5bert-base-uncased__train_full_model__some_labels/checkpoint-4800"
    num_layers = 2
elif model_version == "2_2":
    ckpt = wor_dir + "/results/BERT_models/NACE_classification/037_results__data_approach_3__desc_lvl_level_1_dataset_2__num_layers_1__cos_thres_0.5bert-base-uncased__train_full_model__some_labels__only_labels/checkpoint-743"
    num_layers = 1

model = classification_report_BERT.load_custom_bert_from_checkpoint(ckpt_path=ckpt, num_layers_base=num_layers)
tokenizer = AutoTokenizer.from_pretrained(ckpt)

In [16]:
preprocess_report_bert_size = lambda pdf_path: preprocessing.preprocess_report_into_bert_chunks(pdf_path=pdf_path, tokenizer=tokenizer)

In [17]:
for i in range(1,2):
    nace_level = i
    result_path = wor_dir + f"/results/BERT_classification/model_{model_version}__BERT_preprocessing__desc_pages__with_relevancy__dataset__{dataset_name}_sentence_len_{sentence_length}__nace_level_{nace_level}"
    os.makedirs(result_path, exist_ok=True)
    config = {"model": ckpt, "dataset": dataset_name}
    with open(os.path.join(result_path, "config.json"), "w") as f: 
        json.dump(config, f)

    res = test_base.test_report_classification(
        reports_path=reports_path,
        preprocess_report=preprocess_report_bert_size, 
        report_to_nace_class=report_to_nace_class, 
        result_path = result_path,
        level=i,
        overwrite=True, 
        classification_function=classification_report_BERT.classify_report, 
        path_nace_code_descriptions="data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", 
        model=model,
        tokenizer=tokenizer)

  0%|                                                                                                                                                                                          | 0/63 [00:00<?, ?it/s]

data/datasets/stoxx_600/company_descriptions_txt/AAK AB1.txt


  2%|██▊                                                                                                                                                                               | 1/63 [00:01<01:10,  1.14s/it]

data/datasets/stoxx_600/company_descriptions_txt/ABB Ltd.2.txt


  5%|████████▍                                                                                                                                                                         | 3/63 [00:01<00:24,  2.42it/s]

...

 97%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍     | 61/63 [00:18<00:00,  3.69it/s]

data/datasets/stoxx_600/company_descriptions_txt/Siemens Energy AG1.txt


 98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏  | 62/63 [00:19<00:00,  3.32it/s]

data/datasets/stoxx_600/company_descriptions_txt/Signify NV3.txt


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:19<00:00,  3.25it/s]


In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))   # subtract max for numerical stability
    return e / e.sum()


In [ ]:
df = pd.read_csv("results/BERT_classification/model_1_1__desc_pages__with_relevancy__dataset__stoxx_600_sentence_len_6__nace_level_1/Siemens Energy AG1.txt/Siemens Energy AG1.txt_classifications.csv")

In [ ]:
scores = ['K', 'P', 'I', 'F', 'L', 'J', 'E', 'Q', 'B', 'D', 'A', 'H', 'M', 'G', 'C']

In [ ]:
df.loc[:,"classification"] = [str(row.sort_values(ascending=False).to_dict()) for i, row in df[scores].apply(softmax, axis=1).iterrows()]

In [ ]:
df.loc[0,"classification"]

"{'M': 0.9923605928892378, 'F': 0.0029998710186998974, 'C': 0.0009449491860713078, 'K': 0.0008007685486060554, 'J': 0.0004997021788788219, 'G': 0.000495891890189294, 'P': 0.00046945749140778865, 'Q': 0.0003521442087661222, 'E': 0.0002670324855598185, 'I': 0.00025786005714058057, 'D': 0.00018081801673530733, 'B': 0.00016571749330049446, 'H': 0.0001286617154597945, 'L': 6.741033246666465e-05, 'A': 9.122487480155394e-06}"